# Automatic Deep Research 

Welcome to this new practice lab! By now you should have a clearer view of the elements that compose a multi-agent system. In this lab you will get to put it into action by creating your first crew.

**What you'll learn:**
- How to define agents with specific roles and expertise
- How to provide agents with tools to perform their tasks
- How to create your own tasks that agents will execute
- How to assemble agents and tasks into a Crew, all using CrewAI

## Background

As a research consultant, you're constantly tasked with producing comprehensive reports on diverse topics for demanding clients. You need to build an automatic deep research solution that can rapidly gather, verify, and synthesize information from across the internet, delivering reliable, fact-checked reports that meet tight deadlines and exacting standards regardless of the subject matter. 

## General instructions
In this lab you will be presented with a structure of the code, but you will need to complete some of it. 

To successfully run this lab, replace all instances of the placeholder `None` with your own code. Sections where you need to write code will be delimited between `### START CODE HERE ###` and `### END CODE HERE ###`.

If you are stuck, or simply want to copy a solution into your notebook so that you can execute it, you can find all solution code inside the [Solution](Solution) folder.

**<font color='#5DADEC'>Please make sure to save your work periodically, so you don't lose any progress.</font>**

## Table of contents

- [1. Understanding the problem](#1)
- [2. Set up your notebook](#2)
- [3. Define the Agents](#3)
  - [3.1. Create tool instances](#3-1)
  - [3.2. Define the Research Planner agent](#3-2)
  - [3.3. Define the remaining agents](#3-3)
- [4. Create the Tasks](#4)
  - [4.1. Define the Create research plan task](#4-1)
  - [4.2. Define the remaining tasks](#4-2)
- [5. Define the Crew and get the results](#5)

<a id="1"></a>

## 1. Understanding the problem
In this lab, you will focus on building a custom deep research crew. This Crew will be in charge of creating a research plan based on the user's input, and executing it, while reviewing and checking the facts. Finally, with the gathered information a report needs to be generated.

Take some time to decompose the problem into different tasks. Who would be the appropriate "person" to solve each task? 

Once you've done your thinking, click below to find an agent/task diagram for this lab.    


<details>    
<summary>
    <font size="3" color="#237b946b"><b>Diagram</b></font>
</summary>

<img src="../images/lab2-agents-tasks-diagram.PNG">

<a id="2"></a>

## 2. Set up your notebook

Before you start coding, run the next two cells to import all necessary modules and configure the environment variables. 

In [17]:
from crewai import Agent, Task, Crew
import os
from datetime import date

TODAY = date.today().isoformat()

In [11]:
from crewai import Agent, Task, Crew
import os
from datetime import date
os.environ["CREWAI_TESTING"] = "true"
from utils import get_openai_api_key

# set the OpenAI model (gpt-4o-mini)
os.environ["MODEL"] = "gpt-4o-mini"
# set up the OpenAI API key
os.environ["OPENAI_API_KEY"] = get_openai_api_key()

# Today's date is injected into the tasks so the agents always know
# what "recent" / "latest" / "this week" actually means when they search.

print("Model:", os.environ["MODEL"])
print("Today's date (used for recency grounding):", TODAY)

Model: gpt-4o-mini
Today's date (used for recency grounding): 2026-08-15


In [23]:
!pip install "crewai[google-genai]" -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import google.genai
print("Google GenAI installed")

Google GenAI installed


In [ ]:
import os

os.environ["GEMINI_API_KEY"] = ""

In [20]:
from crewai import Agent, LLM

llm = LLM(
    model="gemini/gemini-2.5-flash",
    temperature=0
)

<a id="3"></a>

## 3. Define the Agents

Based on the diagram, you should have four agents:
- **Research Planner**: its goal is to analyze queries and break them down into smaller, specific research topics.
- **Internet Researcher**: its job is to perform research tasks.
- **Fact checker**: its goal is to review information for fact accuracy to avoid misinformation. 
- **Report Writer**: is in charge of writing reports, based on gathered information.

<a id="3-1"></a>

### 3.1. Create tool instances
As you can see in the diagram, you will be providing the **Internet Researcher Agent** with tools, so that it can better do their job. In particular, you will give this agent access to search the internet and scrape information from the retrieved webpages. 

There are different tools inside CrewAI you can use to search the web, in this lab you will use the [**EXA Search Web Loader**](https://docs.crewai.com/en/tools/search-research/exasearchtool#exa-search-web-loader) tool, which is designed to perform a semantic search for a specified query from a text’s content across the internet. It utilizes the [exa.ai](https://exa.ai/) API to fetch and display the most relevant search results based on the query provided by the user. exa.ai enhances semantic search by capturing richer contextual relationships between concepts, allowing for more precise information retrieval than conventional embedding approaches.

For webscraping, you will use the [**Scrape Website**](https://docs.crewai.com/en/tools/web-scraping/scrapewebsitetool) tool, which is designed to extract and read the content of a specified website.

In the next cell you will define instances of these tools, so you can later assign them to the agents.

In [21]:
# import the tools
from crewai_tools import EXASearchTool, ScrapeWebsiteTool
from utils import get_exa_api_key

# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

# Create the EXASearchTool instance (web + news semantic search)
exa_search_tool = EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))
# Create the ScrapeWebsiteTool instance (full-page content extraction)
scrape_website_tool = ScrapeWebsiteTool()

<a id="3-2"></a>

### 3.2. Define the Research Planner agent

In the cell below, you will see how you can create the first agent. This time, all the parameters are set up for you. Here is a quick recap of what each of the parameters represent:

- `Role`: If this was a person doing the job, what title would they have?
- `Goal`: What is the goal this agent in particular is trying to accomplish? Make sure to write concrete goal
- `Background`: it should be something the highlights the skills of the agent relevant to its role. Make sure to use keywords that will actually help your agent get better results.

In the labs, we have added two parameters not shown in the demo videos: `max_rpm`, and `max_iter`. `max_rpm` sets the maximum requests per minute to avoid rate limits, while `max_iter` limits the maximum iterations before the agent must provide its best answer. Setting these two parameters helps make the agents run a little faster, so the lab doesn't take as long to complete. 

In [ ]:
query_analyzer = Agent(
    role="Fact-Check Query Analyzer",
    goal=(
        "Turn the user's query or claim into a precise, checkable research brief in one pass: "
        "list the specific sub-claims, name the exact entities/events/dates involved, classify the "
        "query as HISTORICAL (a settled past event, historical figure, or long-standing claim/myth "
        "-- e.g. 'Hitler had only one testicle') or CURRENT (recent/ongoing, needs today's or this "
        "week's information), and set the correct time window accordingly -- a fixed historical period "
        "for historical claims, or 'as of {current_date}' for current ones. Output only the brief, "
        "no extra commentary, to keep the pipeline fast and cheap."
    ),
    backstory=(
        "You are a fact-check desk editor (TN Fact Check / TN IID style) who triages incoming claims "
        "before anyone starts searching. Your first instinct is always 'is this about the past or the "
        "present?' -- because that decides whether the researcher should chase this week's news or dig "
        "into archives, encyclopedias, and historical scholarship instead. You never send a vague brief "
        "downstream: every sub-claim you produce is specific enough to search verbatim. You write in "
        "tight, structured lists, never prose, to keep the brief short and unambiguous."
    ),
    verbose=True,
    max_rpm=150,
    max_iter=8,
    llm=llm
)

<a id="3-3"></a>

### 3.3. Define the remaining agents

Now you can define the three remaining agents. The `role` and `goal` parameters are already filled in for you; use your own creativity to fill in the `backstory`.  

Do not forget to assign the tools to the **Internet Researcher** and **Fact Checker** agents. You can do this by setting the `tools` argument.

In [ ]:
researcher = Agent(
    role="Research Investigator",
    goal=(
        "For CURRENT sub-claims: retrieve the most recent relevant information, prioritizing breaking "
        "news, official statements, and reports inside the required time window, using recency-biased "
        "search terms ('latest', 'today', the specific month/year). "
        "For HISTORICAL sub-claims: retrieve information from authoritative historical sources -- "
        "encyclopedic references, archives/museums, academic or established journalistic accounts -- "
        "and, when the claim resembles a known myth or urban legend, specifically search for its origin "
        "and any documented debunking. In both cases, record the exact source title, URL, publisher, "
        "and publish/last-updated date for every finding, and stop once you have 3-5 solid sources per "
        "sub-claim -- do not over-search, since every extra call costs time and tokens."
    ),
    backstory=(
        "You are an investigative researcher equally comfortable with a live newswire and a library "
        "archive. For breaking stories you search with urgency and recency bias, cross-checking "
        "multiple outlets. For historical claims you reach for primary and archival material and "
        "recognized reference sources rather than blogs or forums, since old claims are especially "
        "prone to myth and repetition without scrutiny. You scrape a page only when a snippet can't "
        "confirm a date, figure, or quote, and you always summarize findings in 1-2 sentences rather "
        "than pasting long passages of text."
    ),
    tools=[exa_search_tool, scrape_website_tool],
    verbose=True,
    max_rpm=150,
    max_iter=10,
    llm=llm
)

fact_verifier = Agent(
    role="Fact Verification Specialist",
    goal=(
        "Independently verify each finding against at least one additional credible source beyond "
        "what the researcher already found -- a second outlet for current claims, or a second "
        "reputable historical/academic source for historical claims. Flag anything outdated, "
        "unsupported, contradictory, or resting on a single source. Rate each source's reliability "
        "(High/Medium/Low) and, for current claims, its recency. Assign each sub-claim a status: "
        "Verified, Partially Verified, Unverified, Outdated, or False."
    ),
    backstory=(
        "You are a meticulous verification specialist in the TN Fact Check / TN IID mold. You never "
        "accept a single source at face value -- you re-search and cross-check, treating persistent "
        "historical myths and rumors with exactly the same scrutiny as breaking-news hoaxes. You keep "
        "your reasoning tight: one clear justification per sub-claim, never a running commentary."
    ),
    tools=[exa_search_tool, scrape_website_tool],
    verbose=True,
    max_rpm=150,
    max_iter=10,
    llm=llm
)

factcheck_report_writer = Agent(
    role="Fact-Check Report Writer",
    goal=(
        "Convert the verified findings into the final structured fact-check report that will be "
        "rendered directly in a frontend UI: an overall verdict, a short executive summary, a "
        "per-sub-claim breakdown, and a complete, deduplicated source list (URL, publisher, date, "
        "reliability) for every source used. Be precise and concise -- the frontend has limited "
        "space, so avoid filler, hedging, or repeating the same source description twice."
    ),
    backstory=(
        "You are a professional fact-check report writer who turns a verification desk's findings "
        "into a publication-ready, UI-ready verdict -- similar to how TN Fact Check / TN IID present "
        "their conclusions. You always lead with the verdict, never bury it, and every source you cite "
        "is traceable to a real URL gathered earlier in the pipeline -- you never invent a source."
    ),
    verbose=True,
    max_rpm=150,
    max_iter=6,
    llm=llm
)

<a id="4"></a>

## 4. Create the Tasks

Now that you have set up the agents, it is time to define the tasks. If you go back to the diagram, you will see you need four tasks:

- **Create research plan**: Based on the user's query, break it down into specific topics and key questions, and create a focused research plan.
    - Output: A research plan with main research topics to investigate, key questions for each topic, and success criteria for the research.

- **Gather research data**: Using the research plan, collect information on all identified topics. Cite all sources used.
    - Output: Comprehensive research data including: information for each research topic, and citations used along with source credibility notes.

- **Verify information quality**: Review all collected research. Identify any conflicting information, potential misinformation, or gaps that need addressing.
    - Output: A report with the all the collected data, and its review. It should include consistency check results and source reliability ratings

- **Write final report**: Create a comprehensive report that answers the original query using all verified research data. Structure it with clear sections, include citations, and provide actionable insights.
    - Output: The final research report. In addition to the full answer, it should have an executive summary, and complete source citations.


For each `Task` you need to define the following parameters:
- `description`: A thorough description of the task. You can even break it down into different items.
- `expected_output`: what should the output return. Be specific, specially if you want any structure in your result, like a dictionary with specific keys.
- `agent`: who is performing the task? You need to match the task to one of the agents you already defined

In the description you will need to pass the inputs to the tasks. In this lab, you will only have as input the user's query, which will be saved as `user_query`:


<a id="4-1"></a>

### 4.1. Define the Create research plan task

In the cell below, you will see how you can create the first task. This time, all the parameters are set up for you. Notice how the context variables are passed the the description between curly brackets. 

### 3.4. Define the structured output schema

Since the final report will be rendered directly in a frontend, the last task should not return free-form text -- it should return a well-defined JSON object. The cell below defines that schema with Pydantic. `crewAI` will force the final agent's output to match this structure (via `output_pydantic`), which also saves tokens because the model doesn't need to "explain itself" in prose -- it just fills in fields.

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional


class SourceCitation(BaseModel):
    title: str = Field(description="Title of the source article/page")
    url: str = Field(description="Direct URL of the source")
    publisher: str = Field(description="Publication, organization, or site name")
    published_date: Optional[str] = Field(
        default=None,
        description="Publish or last-updated date (YYYY-MM-DD if known, else 'unknown')"
    )
    reliability: str = Field(description="High, Medium, or Low")
    supports: str = Field(description="Confirms, Contradicts, or Partially confirms the sub-claim")


class SubClaimResult(BaseModel):
    sub_claim: str = Field(description="The specific sub-claim being checked")
    status: str = Field(description="Verified, Partially Verified, Unverified, Outdated, or False")
    explanation: str = Field(description="1-3 sentence plain-language explanation of the status")
    sources: List[SourceCitation] = Field(description="Sources supporting this sub-claim's status")


class FactCheckReport(BaseModel):
    user_query: str = Field(description="The original user query/claim")
    is_historical: bool = Field(
        description="True if the claim is about a settled past event/figure, False if current/ongoing"
    )
    time_window_used: str = Field(description="The time window/period the research was grounded in")
    overall_verdict: str = Field(
        description="True, False, Misleading, Unverified, or Needs More Context"
    )
    confidence: str = Field(description="High, Medium, or Low confidence in the verdict")
    executive_summary: str = Field(description="A 2-4 sentence plain-language summary of the verdict and why")
    sub_claims: List[SubClaimResult] = Field(description="Per sub-claim verification breakdown")
    all_sources: List[SourceCitation] = Field(
        description="Deduplicated list of every source used across the whole investigation"
    )
    report_generated_on: str = Field(description="Date the report was generated")

In [ ]:
analyze_query_task = Task(
    description=(
        "Analyze the user's query and break it down into specific, checkable sub-claims. Identify "
        "all entities, events, organizations, and dates involved. Classify the query as HISTORICAL "
        "(a settled past event, historical figure, or long-standing claim/myth -- e.g. a claim about "
        "WWII or a historical rumor) or CURRENT (recent/ongoing, needs up-to-date information). "
        "Determine the time window: for HISTORICAL claims use the relevant historical period; for "
        "CURRENT claims, if the user specified a date/period use exactly that, otherwise target as of "
        "{current_date}. Produce concrete search queries for each sub-claim: recency-biased terms for "
        "current claims (e.g. 'latest', 'today', the relevant year/month), or precise historical/"
        "reference terms for historical claims (e.g. exact names/dates, 'origin of', 'myth').\n\n"
        "The user's query is: {user_query}\n"
        "The user-specified time period (if any) is: {time_period}\n"
        "Today's date is: {current_date}"
    ),
    expected_output=(
        "A concise research brief listing: (1) the specific sub-claims to verify, (2) the exact "
        "entities/events/dates involved, (3) whether the query is HISTORICAL or CURRENT and the "
        "determined time window, and (4) 1-3 concrete search queries per sub-claim. No extra prose."
    ),
    agent=query_analyzer,
)

<a id="4-2"></a>

### 4.2. Define the remaining tasks

Now define the three remaining tasks. The `description` is already filled in for you, you will need to define the `expected_output` and `agent` for each of the Tasks.

In [ ]:
# define the gather evidence task
gather_evidence_task = Task(
    description=(
        "Using the research brief, search for evidence on every sub-claim. For CURRENT sub-claims, "
        "prioritize sources published within the determined time window using recency-biased search "
        "terms. For HISTORICAL sub-claims, prioritize encyclopedic, archival, academic, or established "
        "reference sources over blogs/forums, and specifically look for the origin and any documented "
        "debunking if the claim resembles a known myth. For every finding, record the source title, "
        "URL, publisher, and publish/last-updated date (or 'unknown' if undated). Gather 3-5 solid "
        "sources per sub-claim -- stop once you have enough to verify, do not over-search. Scrape a "
        "page only when the snippet doesn't confirm a date, figure, or quote."
    ),
    expected_output=(
        "For each sub-claim: the finding in 1-2 sentences, plus a short list of sources (title, URL, "
        "publisher, date). Clearly mark whether each sub-claim was treated as HISTORICAL or CURRENT."
    ),
    agent=researcher
)

# define the verify facts task
verify_facts_task = Task(
    description=(
        "Review all gathered evidence. For each sub-claim, verify it against at least one additional "
        "independent, credible source beyond what the researcher already found. Identify conflicting "
        "information, outdated claims (superseded by newer developments, for CURRENT sub-claims), or "
        "myths/misinformation (for HISTORICAL sub-claims). Check that each source is appropriate to "
        "the claim's era -- recent, dated sources for current claims; reputable historical/academic "
        "sources for historical ones. Assign each sub-claim a status: Verified, Partially Verified, "
        "Unverified, Outdated, or False, and rate each source's reliability (High/Medium/Low)."
    ),
    expected_output=(
        "For each sub-claim: a status (Verified/Partially Verified/Unverified/Outdated/False), a "
        "1-3 sentence justification, and the sources used with a reliability rating each. Keep it "
        "tight -- no repeated explanations."
    ),
    agent=fact_verifier
)

# define the write final fact-check report task
write_factcheck_report_task = Task(
    description=(
        "Using the verified findings, produce the final fact-check report as structured data for "
        "direct display in a frontend UI. Include: the original user query; whether the claim is "
        "historical or current and the time window used; an overall verdict (True / False / "
        "Misleading / Unverified / Needs More Context); a confidence level (High/Medium/Low); a 2-4 "
        "sentence executive summary; a per-sub-claim breakdown with status, a short explanation, and "
        "its sources; and a complete, deduplicated list of every source used across the whole "
        "investigation, each with title, URL, publisher, date, reliability, and whether it confirms/"
        "contradicts/partially confirms the claim. Only cite sources that were actually gathered "
        "earlier in the pipeline -- never invent a source or URL. Today's date is {current_date}."
    ),
    expected_output=(
        "A FactCheckReport object with every field populated: user_query, is_historical, "
        "time_window_used, overall_verdict, confidence, executive_summary, sub_claims (each with "
        "sub_claim, status, explanation, sources), all_sources (deduplicated), and report_generated_on."
    ),
    agent=factcheck_report_writer,
    output_pydantic=FactCheckReport,
)

<a id="5"></a>

## 5. Define the Crew and get the results

Once the agents and tasks have been defined, you are ready to create the crew. In order to so, you will need to set the following arguments:
- `agents`: list of agents in the crew
- `tasks`: list of tasks in the crew. The tasks should be listed in the order they should be executed

In the next cell, fill in the agents and tasks for the crew.

In [ ]:
# create the crew with the defined agents and tasks
crew = Crew(
    agents=[query_analyzer, researcher, fact_verifier, factcheck_report_writer],
    tasks=[analyze_query_task, gather_evidence_task, verify_facts_task, write_factcheck_report_task]
)

Before running the crew, you need to define the query, which will be used as input for the tasks.

In [ ]:
# Write the query/claim you want fact-checked
user_query = "MK stalin won in recounting in kolathur"

# Optional: pin the fact-check to a specific date/period.
# Leave as "not specified" to default to the most recent available information.
time_period = "August 2026"

# --- The pipeline also handles HISTORICAL claims out of the box, e.g.: ---
# user_query = "Hitler had only one testicle"
# time_period = "not specified"   # historical claims don't need a current time window

Now you are only left with kickstarting the crew to get the results. Since you set `verbose=True` in the agents, you should monitor all the process.

In [31]:
result = crew.kickoff(
    inputs={
        "user_query": user_query,
        "time_period": time_period,
        "current_date": TODAY,
    }
)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Query Analyzer                                                                               │
│                                                                                                                 │
│  Task: Analyze the user's query and break it down into specific, checkable sub-claims and key questions.        │
│  Identify all entities, events, organizations, and dates involved. Determine the correct time window for        │
│  research: if the user specified a date or period, use exactly that; otherwise, target the most recent          │
│  information available as of 2026-08-15. Produce a focused research plan with concrete, recency-biased search   │
│  queries (e.g. including terms like 'latest', 'today', 'this week', or the relevant year/month) for the         │
│  researcher to use.                                                                                             │
│                                                                                                                 │
│  The user's query is: MK stalin won in recounting in kolathur                                                   │
│  The user-specified time period (if any) is: August 2026                                                        │
│  Today's date is: 2026-08-15                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Query Analyzer                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Research Plan for "MK stalin won in recounting in kolathur"**                                                │
│                                                                                                                 │
│  **1. Specific Sub-claims/Questions to Verify:**                                                                │
│                                                                                                                 │
│  *   Is there an election (general assembly election or by-election) scheduled or ongoing in the Kolathur       │
│  constituency, Tamil Nadu, in August 2026?                                                                      │
│  *   If an election is confirmed for August 2026, is M.K. Stalin a declared candidate for the Kolathur          │
│  constituency in that election?                                                                                 │
│  *   If M.K. Stalin is a candidate and an election has occurred, has there been a request for or an actual      │
│  recounting of votes in the Kolathur constituency in August 2026?                                               │
│  *   If a recounting occurred, what were the official results, and did M.K. Stalin win the election in          │
│  Kolathur after this recounting in August 2026?                                                                 │
│                                                                                                                 │
│  **2. Exact Entities/Events/Dates Involved:**                                                                   │
│                                                                                                                 │
│  *   **Entities:** M.K. Stalin, Kolathur constituency (Tamil Nadu, India), Election Commission of India (or     │
│  Tamil Nadu State Election Commission), Dravida Munnetra Kazhagam (DMK) party.                                  │
│  *   **Events:** State Assembly Election (or by-election), Candidacy Declaration, Vote Recounting, Election     │
│  Results Announcement, Election Victory.                                                                        │
│  *   **Dates:** Specifically August 2026. The information should be current as of 2026-08-15.                   │
│                                                                                                                 │
│  **3. Determined Time Window to Search Within:**                                                                │
│                                                                                                                 │
│  *   August 2026 (with a focus on news and official announcements from 2026-08-01 to 2026-08-15, and any        │
│  preceding announcements for events occurring in August 2026).                                                  │
│                                                                                                                 │
│  **4. Concrete, Recency-Biased Search Queries for Each Sub-claim:**                                             │
│                                                                                                                 │
│  *   **For Sub-claim 1 (Election in Kolathur, August 2026):**                                                   │
│      *   `"Kolathur election August 2026"`             

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Real-Time News Researcher                                                                               │
│                                                                                                                 │
│  Task: Using the research plan, search the web and news sources for the most recent, relevant information on    │
│  every identified sub-claim and topic. Prioritize sources published within the determined time window -- use    │
│  recency-biased search terms and, when the user gave no specific period, actively seek out the latest           │
│  available news, actions, or developments rather than older background material. For every piece of             │
│  information gathered, record the source URL and its publish date, and scrape the page when needed to confirm   │
│  exact dates, figures, or quotes.                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Real-Time News Researcher                                                                               │
│                                                                                                                 │
│  Thought: Action: EXASearchTool                                                                                 │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {                                                                                                              │
│    "search_query": "\"Kolathur election August 2026\"",                                                         │
│    "start_published_date": "2026-08-01",                                                                        │
│    "end_published_date": "2026-08-15"                                                                           │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Title: Kolathur Election Results 2026 Declared: M.K Stalin looses after 15 years; TVK sweeps DMK's fortress |  │
│  News | Zee News                                                                                                │
│  URL:                                                                                                           │
│  https://zeenews.india.com/india/live-updates/kolathur-election-results-2026-live-updates-mk-stalin-dmk-vs-aia  │
│  dmk-aadirajaram-vs-tvk-vs-babu-3043281.html                                                                    │
│  ID:                                                                                                            │
│  https://zeenews.india.com/india/live-updates/kolathur-election-results-2026-live-updates-mk-stalin-dmk-vs-aia  │
│  dmk-aadirajaram-vs-tvk-vs-babu-3043281.html                                                                    │
│  Score: None                                                                                                    │
│  Published Date: 2026-08-12T00:00:00.000Z                                                                       │
│  Author: None                                                                                                   │
│  Image: https://english.cdn.zeenews.com/sites/default/files/2026/05/04/1938964-kolathur.jpg.jpeg                │
│  Favicon: None                                                                                                  │
│  Extras: None                                                                                                   │
│  Subpages: None                                                                                                 │
│  Text: Kolathur Election Results 2026 Declared: M.K Stalin looses after 15 years; TVK sweeps DMK's fortress |   │
│  News | Zee News                                                                                                │
│                                                                                                                 │
│  # Kolathur Election Results 2026 Declared: M.K Stalin looses after 15 years; TVK sweeps DMK's fortress         │
│                                                                                                                 │
│  ## Kolathur Election Results 2026 Declared: One of Tamil Nadu’s high-stakes seats was locked in a triangular   │
│  battle between DMK, AIADMK and TVK. CM M.K. MK Stalin looses his stronghold after 15 years, TVK’s VS Babu      │
│  sweeps by 8795 vote margin. Stalin won the seat by 41% margin in 2021. DMK looses its bastion after 15 years.  │
│                                                                                                                 │
│  Written By Zee Media Bureau Edited By Anjali Singh                                                             │
│                                                                                                                 │
│  Published: May 03, 2026, 08:17 PM IST| Updated: May 04, 2026, 06:07 PM IST                                     │
│                                                                                                                 │
│  04 May 2026 16:40 IST (IST)                                                                                    │
│                                                                                                                 │
│  ### Kolathur Election Results 2026 Live: MK Stalin looses his den after 15 years; TVKs VS Babu marks historic  │
│  winKolathur Election Results 2026: DMK looses its bastion after 15 years; TVK scripts history. MK Stalin       │
│  trails with massive margin of 8284 vot               

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact Verification Specialist                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│                                                                                                                 │
│  Thought:The search results confirm what the provided research indicated:                                       │
│  *   The "Kolathur Election Results 2026 Declared" in May 2026 (Zee News, published May 3, 2026, updated May    │
│  4, 2026, and also a search result with a published date of 2026-08-12, which seems to be a re-indexing or      │
│  update of the original May article).                                                                           │
│  *   The August 2026 events are consistently referred to as "EVM Checking and Verification process" or "EVM     │
│  verification," not an election or recounting, as explicitly stated by election officials (The Hindu, August    │
│  1, 2026).                                                                                                      │
│  *   The Wikipedia page for "17th Tamil Nadu Assembly" states it was constituted after the "April 2026 State    │
│  Assembly elections" and the term began "11 May 2026".                                                          │
│                                                                                                                 │
│  Therefore, Sub-claim 1 is False. There was no election in Kolathur in August 2026. The election was in         │
│  April/May 2026.                                                                                                │
│                                                                                                                 │
│  Now, let's move to Sub-claim 2.                                                                                │
│                                                                                                                 │
│  **Sub-claim 2: If an election is confirmed for August 2026, is M.K. Stalin a declared candidate for the        │
│  Kolathur constituency in that election?**                                                                      │
│                                                                                                                 │
│  *   Since Sub-claim 1 is False (no election in August 2026), this sub-claim is also False. M.K. Stalin was a   │
│  candidate in the *May 2026* election.                                                                          │
│                                                                                                                 │
│  Let's move to Sub-claim 3.                                                                                     │
│                                                                                                                 │
│  **Sub-claim 3: If M.K. Stalin is a candidate and an election has occurred, has there been a request for or an  │
│  actual recounting of votes in the Kolathur constituency in August 2026?**                                      │
│                                                                                                                 │
│  *   The research and search results consistently refer to an "EVM verification process," not a "recounting."   │
│  *   **Source:** "No evidence of EVM tampering, says poll official on fourth day," The Hindu, Published:        │
│  2026-08-01, URL:                                      

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Report Writer                                                                                │
│                                                                                                                 │
│  Task: Create a final fact-check report that directly answers the user's original query using only the          │
│  verified, up-to-date research. Lead with a clear overall verdict (True / False / Misleading / Unverified /     │
│  Needs More Context). Summarize the supporting evidence in plain language, explicitly state how recent the      │
│  underlying information is (mention specific dates), and list every source used with its link and publish       │
│  date.                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fact-Check Report Writer                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Fact-Check Report: "MK stalin won in recounting in kolathur"**                                               │
│                                                                                                                 │
│  **Overall Verdict: False**                                                                                     │
│                                                                                                                 │
│  **Executive Summary:**                                                                                         │
│  The claim that M.K. Stalin won in a recounting in Kolathur in August 2026 is **False**. The Tamil Nadu         │
│  Assembly election for the Kolathur constituency was held in April 2026, with results declared on May 3-4,      │
│  2026. In that election, M.K. Stalin lost the Kolathur seat to V.S. Babu of the Tamilaga Vettri Kazhagam (TVK)  │
│  by a margin of 8,795 votes. While an "EVM Checking and Verification process" was conducted in late July and    │
│  early August 2026 following allegations of discrepancies by the DMK, election officials explicitly stated      │
│  that this was not a vote recounting. This verification process concluded with officials confirming no          │
│  evidence of tampering or irregularities that would alter the original election outcome. Therefore, M.K.        │
│  Stalin did not win in any recounting in Kolathur in August 2026.                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Detailed Evidence Breakdown:**                                                                               │
│                                                                                                                 │
│  **Sub-claim 1: Is there an election (general assembly election or by-election) scheduled or ongoing in the     │
│  Kolathur constituency, Tamil Nadu, in August 2026?**                                                           │
│  *   **Status:** False                                                                                          │
│  *   **Supporting Evidence:**                                                                                   │
│      *   The Tamil Nadu Assembly election for the Kolathur constituency, where M.K. Stalin was a candidate,     │
│  concluded with results declared on May 3-4, 2026.                                                              │
│      *   The 17th Tamil Nadu Assembly was constituted after the April 2026 State Assembly elections, with its   │
│  term commencing on May 11, 2026.                                                                               │
│      *   News reports from August 2026 consistently refer to an "EVM Checking and Verification process"         │
│  related to the *previous* election, not a new election or by-election. Chennai District Election Officer G.S.  │
│  Sameeran explicitly clarified that the August events constituted an "EVM Checking and Verification process,"   │
│  not a recounting.                                     

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 7862dc93-a28f-434d-851d-c13d1a0e5ab4                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/7862dc93-a28f-434 │
│ d-851d-c13d1a0e5ab4?access_code=TRACE-133ce7efac                             │
│ 🔑 Access Code: TRACE-133ce7efac                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


From the output of the previous cell check all the outputs for each task. Do they match what you expected? If not, go back and refine the `expected_output`. 

You can also print the final report to see the final result of the crew

In [ ]:
import json
from IPython.display import Markdown, display

# The structured object -- this is what you send straight to the frontend
report = result.pydantic
print(json.dumps(report.model_dump(), indent=2))

# Optional: a human-readable rendering for quick review in the notebook
display(Markdown(
    f"### Verdict: {report.overall_verdict}  (confidence: {report.confidence})\n\n"
    f"{report.executive_summary}"
))

You made it to the end of the lab! You can go back and experiment with the goals and backstories of the agents, as well as description and expected outputs of tasks. You can also change the inputs to any research topic you wish. Have fun with it!